# 05 — Model Evaluation & Monitoring

**Sources**: `features.target_labels`, `data/predictions/cv_results.parquet`

Đánh giá walk-forward, phân tích lỗi, SHAP explanations, monitoring drift.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = Path().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')
from src.data.storage.postgres_client import get_engine
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODELS_DIR = PROJECT_ROOT / 'models'
PREDS_DIR  = PROJECT_ROOT / 'data' / 'predictions'
engine = get_engine()
print('[OK] Setup done')

In [ ]:
# ── Load predictions & leaderboard ────────────────────────────────────────
df_preds = pd.read_parquet(PREDS_DIR / 'cv_results.parquet')
leaderboard = pd.read_parquet(PREDS_DIR / 'leaderboard.parquet')

print('=== Leaderboard (saved from 04_modeling) ===')
print(leaderboard.sort_values('RMSE'))
best_model = leaderboard['RMSE'].idxmin()
print(f'\n→ Best model: {best_model}')

In [ ]:
# ── Predicted vs Actual ────────────────────────────────────────────────────
pred_cols = [c for c in df_preds.columns if c != 'y_true']

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_preds.index, df_preds['y_true'], color='goldenrod', lw=1.5, label='Actual', zorder=5)
colors = ['steelblue', 'firebrick', 'seagreen', 'darkorchid']
for col, color in zip(pred_cols[:4], colors):
    ax.plot(df_preds.index, df_preds[col], lw=0.9, alpha=0.75, color=color, label=col)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.set_title('Model Predictions vs Actual Gold Price (Test Set)', fontsize=14)
ax.set_ylabel('Price (USD)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Residual analysis (Best model) ────────────────────────────────────────
if best_model in df_preds.columns:
    residuals = df_preds['y_true'] - df_preds[best_model]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Time series of residuals
    axes[0].plot(df_preds.index, residuals, color='firebrick', lw=0.8)
    axes[0].axhline(0, color='black', lw=0.8)
    axes[0].set_title(f'Residuals over Time ({best_model})')
    axes[0].set_ylabel('Residual (USD)')
    axes[0].grid(True, alpha=0.3)

    # Histogram
    axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(residuals.mean(), color='red', linestyle='--', label=f'Mean: {residuals.mean():.2f}')
    axes[1].set_title('Residual Distribution')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Scatter: predicted vs actual
    axes[2].scatter(df_preds[best_model], df_preds['y_true'], alpha=0.4, s=10, color='steelblue')
    mn, mx = df_preds['y_true'].min(), df_preds['y_true'].max()
    axes[2].plot([mn, mx], [mn, mx], 'r--', lw=1)
    axes[2].set_xlabel('Predicted')
    axes[2].set_ylabel('Actual')
    axes[2].set_title('Predicted vs Actual')
    axes[2].grid(True, alpha=0.3)

    plt.suptitle(f'Residual Analysis — {best_model}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Rolling Walk-Forward Validation ───────────────────────────────────────
# Load full feature set
with engine.connect() as conn:
    df_feat = pd.read_sql(
        'SELECT * FROM features.master_features ORDER BY date',
        conn, index_col='date', parse_dates=['date']
    )
    df_tgt = pd.read_sql(
        """SELECT date, next_1_day_price
           FROM features.target_labels
           WHERE next_1_day_price IS NOT NULL ORDER BY date""",
        conn, index_col='date', parse_dates=['date']
    )

df_all = df_feat.join(df_tgt, how='inner')

# Exclude raw OHLC + targets from features
EXCLUDE = ['gold_open', 'gold_high', 'gold_low', 'gold_volume', 'next_1_day_price', 'updated_at']
feat_cols = [c for c in df_all.columns if c not in EXCLUDE]
df_all = df_all[feat_cols + ['next_1_day_price']].dropna()

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
    tscv = TimeSeriesSplit(n_splits=5)
    wf_results = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(df_all)):
        X_tr = df_all.iloc[train_idx][feat_cols].values
        y_tr = df_all.iloc[train_idx]['next_1_day_price'].values
        X_te = df_all.iloc[test_idx][feat_cols].values
        y_te = df_all.iloc[test_idx]['next_1_day_price'].values

        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_te_s = sc.transform(X_te)

        m = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                              n_jobs=-1, random_state=42, verbosity=0)
        m.fit(X_tr_s, y_tr)
        pred = m.predict(X_te_s)

        mae  = mean_absolute_error(y_te, pred)
        rmse = np.sqrt(mean_squared_error(y_te, pred))
        mape = np.mean(np.abs((y_te - pred) / y_te)) * 100
        wf_results.append({'fold': fold+1, 'MAE': mae, 'RMSE': rmse, 'MAPE%': mape,
                           'train_n': len(train_idx), 'test_n': len(test_idx)})
        print(f'Fold {fold+1}: MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%')

    wf_df = pd.DataFrame(wf_results)
    print(f'\n=== Walk-Forward Summary ===')
    print(wf_df.describe().T[['mean', 'std', 'min', 'max']])
except ImportError:
    print('[SKIP] xgboost not installed')

In [ ]:
# ── SHAP Explanations ──────────────────────────────────────────────────────
try:
    import shap
    import xgboost as xgb

    # Load best model
    xgb_optuna_path = MODELS_DIR / 'xgboost_optuna.json'
    xgb_path = MODELS_DIR / 'xgboost.json'
    model_path = xgb_optuna_path if xgb_optuna_path.exists() else xgb_path

    m = xgb.XGBRegressor()
    m.load_model(str(model_path))

    scaler = joblib.load(MODELS_DIR / 'scaler.joblib')

    # Use last 1000 rows for SHAP (speed)
    X_shap = df_all[feat_cols].tail(1000).values
    X_shap_sc = scaler.transform(X_shap)

    explainer = shap.TreeExplainer(m)
    shap_values = explainer.shap_values(X_shap_sc)

    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_shap_sc, feature_names=feat_cols,
                      max_display=20, show=False)
    plt.title('SHAP Feature Importance (Top 20)', fontsize=14)
    plt.tight_layout()
    plt.show()
    print('[OK] SHAP done')
except (ImportError, FileNotFoundError) as e:
    print(f'[SKIP] SHAP: {e}')

In [ ]:
# ── Data Drift Monitoring ──────────────────────────────────────────────────
# So sánh phân phối feature giữa train và test
with engine.connect() as conn:
    df_feat_all = pd.read_sql(
        'SELECT * FROM features.master_features ORDER BY date',
        conn, index_col='date', parse_dates=['date']
    )

n_total = len(df_feat_all)
split_idx = int(n_total * 0.80)
df_train_mon = df_feat_all.iloc[:split_idx]
df_test_mon  = df_feat_all.iloc[split_idx:]

monitor_cols = ['gold_close', 'rsi_14', 'dxy_close', 'us_10y_yield', 'real_yield', 'vix']
monitor_cols = [c for c in monitor_cols if c in df_feat_all.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, monitor_cols):
    tr_data = df_train_mon[col].dropna()
    te_data = df_test_mon[col].dropna()
    ax.hist(tr_data, bins=40, alpha=0.5, color='steelblue', density=True, label='Train')
    ax.hist(te_data, bins=40, alpha=0.5, color='firebrick', density=True, label='Test')
    ax.set_title(col)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature Distribution: Train vs Test (Data Drift Check)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Most Recent Predictions ────────────────────────────────────────────────
with engine.connect() as conn:
    df_recent = pd.read_sql(
        """SELECT f.date, f.gold_close, t.next_1_day_price, t.next_7_day_price, t.next_30_day_price
           FROM features.master_features f
           JOIN features.target_labels t ON f.date = t.date
           WHERE t.next_1_day_price IS NOT NULL
           ORDER BY f.date DESC
           LIMIT 10""",
        conn, index_col='date', parse_dates=['date']
    )

print('=== Most Recent Target Labels ===')
print(df_recent)

# Latest row with NULL next prices = most recent trading day
with engine.connect() as conn:
    df_latest = pd.read_sql(
        """SELECT f.date, f.gold_close, t.next_1_day_price
           FROM features.master_features f
           JOIN features.target_labels t ON f.date = t.date
           ORDER BY f.date DESC
           LIMIT 5""",
        conn, index_col='date', parse_dates=['date']
    )

print('\n=== Latest 5 rows (NULL next_price = future) ===')
print(df_latest)